# J2-05 · Bilan du jour : une conversation complète, regroupée en session

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 2 · créneau 15h45–16h30 · ⏱️ 45 min (**démo formateur**, débrief, quiz oral de 5 min)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Assembler les briques du jour : persistance, validation humaine, sortie structurée, robustesse · 2. Suivre une conversation **multi-tours** dans les **sessions** Langfuse · 3. Pseudonymiser l'utilisateur · 4. Vérifier ses acquis |
| 🧱 **Fil rouge** | Le jalon J2 : `helpdesk_ops.jalons.j2` (corrigés des exercices 2.1 et 2.2 inclus) |
| ✅ **Prérequis** | J2-01 à J2-04 |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Le jalon J2 : tout le jour 2 dans un graphe

```
START → garde_fou ─SUPPORT─→ agent ⇄ tools
                  │            └─(réponse)→ qualifier ─pas de ticket─→ END
                  │                                  ├─P2 à P4──────→ creer_ticket → END
                  │                                  └─P1─→ validation_humaine ─approuver/modifier→ creer_ticket
                  │                                                         └─rejeter──────────→ ticket_refuse → END
                  └─sinon──→ refus_poli → END
```

| Brique du jour | Où dans le jalon |
|---|---|
| Persistance (J2-01) | `AsyncSqliteSaver`, `thread_id` = conversation |
| Validation humaine (J2-02) | `validation_humaine` (`interrupt`), `creer_ticket` idempotent |
| Sortie structurée (J2-03) | `qualifier` → `TicketDecision`, routage par `Command(goto=…)` |
| Robustesse (J2-04) | retry + timeout + `error_handler` sur `agent` et `tools` |

In [ ]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from helpdesk_ops.jalons.j2 import arepondre, areprendre, construire_support_j2
from helpdesk_ops.persistance import chemin_checkpoints
from helpdesk_ops.reseau import RESEAU
from helpdesk_ops.ticketing import TICKETING

connexion = await aiosqlite.connect(chemin_checkpoints("j2_05_checkpoints.sqlite"))
saver = AsyncSqliteSaver(connexion)

TIMEOUT_LLM_S = 300   # poste CPU de préparation ; en salle GPU : 30 à 60
helpdesk = construire_support_j2(llm, checkpointer=saver, timeout_llm_s=TIMEOUT_LLM_S)
RESEAU.configurer(taux_echec=0.3, graine=11)   # une supervision capricieuse, comme dans la vraie vie
TICKETING.vider()

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{helpdesk.get_graph().draw_mermaid()}\n```"))

## 3. Pseudonymiser l'utilisateur

### [Théorie] `user_id` : un identifiant stable, pas une identité

Langfuse regroupe les traces par `user_id` : pratique pour suivre un utilisateur, dangereux si c'est son e-mail. On utilise un **pseudonyme stable** : une empreinte salée de l'identifiant. Le sel reste côté application ; sans lui, impossible de remonter à la personne depuis Langfuse.

> 💡 **Le « Tips » du Formateur :** pseudonymiser n'est pas anonymiser. Au sens du RGPD, une donnée pseudonymisée **reste** une donnée personnelle ; mais une fuite de l'UI Langfuse n'expose plus directement des e-mails.

In [ ]:
import hashlib
import os


def pseudonyme(identifiant: str) -> str:
    """Empreinte salée et stable : même personne → même pseudonyme ; le sel ne quitte pas l'application."""
    sel = os.environ["SALT"]   # généré par generate_env.py (aussi utilisé par Langfuse : en production, un sel dédié)
    return "collab-" + hashlib.sha256(f"{sel}:{identifiant.lower()}".encode()).hexdigest()[:10]


UTILISATEUR = pseudonyme("julie.martin@nordtech.example")
CONVERSATION = "j2-05-conv-julie"
print("user_id envoyé à Langfuse :", UTILISATEUR)

## 4. La conversation

Trois tours, une seule session Langfuse. Le premier déclenche une **validation humaine** (P1).

In [ ]:
async def tour(question: str) -> dict:
    with propagate_attributes(session_id=CONVERSATION, user_id=UTILISATEUR, tags=["jour-2", "bilan"]):
        r = await arepondre(helpdesk, question, thread_id=CONVERSATION, callbacks=[langfuse_handler])
    print(f"👤 {question}\n🤖 {r['reponse']}")
    if r["interruption"]:
        print("⏸️  validation demandée :", r["interruption"]["ticket_propose"])
    return r


r = await tour("srv-erp-02 ne répond plus du tout et toute la compta est bloquée depuis 9 h !")

In [ ]:
# Le responsable N2 valide depuis son interface (ici : directement)
if r["interruption"]:
    with propagate_attributes(session_id=CONVERSATION, user_id=UTILISATEUR, tags=["jour-2", "bilan", "validation"]):
        r = await areprendre(helpdesk, {"action": "approuver", "par": "resp-n2"}, thread_id=CONVERSATION,
                             callbacks=[langfuse_handler])
    print("🤖", r["reponse"])

In [ ]:
r = await tour("Merci. Autre chose : comment je reconfigure le client VPN sur mon portable ?")
print("\nTickets :", [(t.id, t.priorite, t.valide_par) for t in TICKETING.lister()])
langfuse.flush()

**👀 Ce que vous devez observer :**
- tour 1 : `ping_serveur` (avec peut-être un retry : la supervision est capricieuse), une réponse, puis la **qualification P1** et la pause ;
- la reprise crée **un** ticket, validé par `resp-n2` ;
- tour 3 : une procédure tirée de la KB ; la qualification ne porte que sur **ce tour** (la question VPN), pas sur l'incident ERP du tour 1 : pas de second ticket ;
- ⚠️ la recherche `chercher_kb` du jour 1 compare des mots : il lui arrive de renvoyer un article hors sujet (pendant la préparation, la procédure VPN pour une panne d'ERP), que l'agent recopie. Le vrai RAG sur Qdrant (J3) et la mesure de fidélité aux sources (J4) traitent ce défaut ;
- dans Langfuse, **Sessions → `j2-05-conv-julie`** : toutes les traces de la conversation, dans l'ordre, avec l'utilisateur **pseudonymisé**. Ouvrez la trace du tour 1 : garde-fou, agent, outils, qualification, tout y est.

## 5. Ce qu'on retient du jour 2

| Brique | La phrase à retenir |
|---|---|
| Persistance | un checkpoint par étape, un `thread_id` par conversation ; et des données personnelles **en clair** à purger |
| Validation humaine | l'agent propose, l'humain dispose ; rien avant `interrupt()`, tout est **idempotent** |
| Sortie structurée | un contrat Pydantic pour router ; mesurer la méthode (`function_calling` ici) et la qualité |
| Robustesse | retry ciblé, timeout async, repli avec `goto` : l'utilisateur reçoit toujours une réponse |
| Observabilité | une conversation = une session ; un utilisateur = un pseudonyme |

### ✅ Quiz du jour

1. Quelle est la différence entre `thread_id` et `session_id` dans notre architecture, et pourquoi les aligner ?
2. Un nœud appelle une API de paiement puis `interrupt()`. Quel est le risque, et comment le corriger ?
3. Pourquoi `TicketDecision` utilise-t-il `Literal["P1", "P2", "P3", "P4"]` plutôt que `str` ?
4. La supervision met 2 minutes à répondre. Que se passe-t-il avec notre jalon J2 ?
5. Julie demande l'effacement de ses données. Citez trois endroits où agir.
6. Pourquoi un `user_id` en clair (l'e-mail) est-il une mauvaise idée ?

<details>
<summary>Réponses</summary>

1. `thread_id` adresse les checkpoints LangGraph ; `session_id` regroupe les traces Langfuse. Alignés, une conversation se retrouve à l'identique dans les deux outils (débogage, audit, effacement).
2. Le paiement est rejoué à la reprise (double débit). Déplacer l'appel après la validation, dans un nœud séparé, avec une clé d'idempotence.
3. Le modèle ne peut pas inventer « P0 » ou « urgent » : Pydantic rejette la sortie, et le routage ne voit que des valeurs connues.
4. Le nœud `tools` coupe chaque tentative au bout de 10 s ; le timeout n'est pas réessayé ; `outils_en_echec` renvoie une erreur au modèle, qui l'annonce à l'utilisateur.
5. Les checkpoints (`delete_thread`), les traces Langfuse (session et utilisateur), le ticketing (résumé), les sauvegardes et journaux.
6. L'UI Langfuse est partagée (Ops, développeurs) : l'e-mail y identifie directement la personne. Un pseudonyme salé limite l'exposition.
</details>

In [ ]:
await connexion.close()
langfuse.flush()
print("✅ Fin du jour 2")